# Module 4: Diagnostics Baseline

## Overview

**This notebook teaches "baseline first" discipline.** Before introducing failures or debugging issues, you must capture what "good" looks like. This baseline becomes your reference point for all troubleshooting.

**⚠️ SAFETY NOTICE:** This notebook is **READ-ONLY**. However, Module 4 failure labs will modify your environment. Ensure you've completed the safety check in `../shared/00_setup_or_resume_environment.ipynb` before proceeding.

**What This Notebook Does:**
1. Captures cluster state snapshot (pods, services, deployments)
2. Collects recent events and resource usage
3. Runs the canonical diagnostics script
4. Performs basic health checks
5. Saves everything to a timestamped directory

**Why This Matters:**
- You need "before" to compare to "after"
- Support will ask for baseline diagnostics
- Good debugging starts with understanding normal state
- Evidence collection is time-sensitive

**Estimated time:** 15-20 minutes

**Important:** 
- Run this notebook BEFORE starting any failure labs. It's your evidence baseline.
- This notebook is read-only and safe to run.


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path
from datetime import datetime

# Add notebooks directory to path so we can import shared as a package
possible_paths = [
    Path.cwd().parent,  # If cwd is module-4, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])

# Create timestamped directory for this baseline
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
baseline_dir = artifacts_dir / "module-4" / f"baseline-{timestamp}"
baseline_dir.mkdir(parents=True, exist_ok=True)

print(f"\nBaseline directory: {baseline_dir}")
print(f"All diagnostics will be saved here.")


## Safety Check: Environment Verification

Verify you're in a safe environment before collecting baseline. Module 4 failure labs will modify your environment.


In [ ]:
# Safety check: Verify environment is safe for Module 4
from shared._cloud_helpers import get_cloud_provider, get_region, get_identity
from shared._validation import ok, warn
import os

provider = get_cloud_provider()
region = get_region()
identity = get_identity()

print("### Environment Safety Check\n")

# Show environment details
provider_display = provider.upper()
print(f"Cloud Provider: {provider_display}")
print(f"Region: {region}")

if provider == "aws":
    print(f"Account ID: {identity.get('Account', 'N/A')}")
    print(f"User ARN: {identity.get('Arn', 'N/A')}")
elif provider == "azure":
    subscription_id = identity.get("SubscriptionId") or identity.get("Account", "N/A")
    print(f"Subscription ID: {subscription_id}")

# Show environment variables
print(f"\n### Environment Variables")
print(f"NAMESPACE: {os.environ.get('NAMESPACE', 'NOT SET')}")
print(f"CLUSTER_NAME: {os.environ.get('CLUSTER_NAME', 'NOT SET')}")
print(f"HELM_RELEASE: {os.environ.get('HELM_RELEASE', 'langsmith')}")

# Check for Module 4 safety flag
module4_safe = os.environ.get("MODULE4_SAFE_ENVIRONMENT", "").lower()
if module4_safe in ["true", "yes", "1"]:
    ok("MODULE4_SAFE_ENVIRONMENT flag is set")
    print("   ✅ Environment verified as safe for Module 4 failure labs")
else:
    warn("MODULE4_SAFE_ENVIRONMENT flag is NOT set")
    print("   💡 This notebook is read-only, but failure labs require this flag")
    print("   💡 Set MODULE4_SAFE_ENVIRONMENT=true in your .env file")
    print("   💡 Complete safety check in ../shared/00_setup_or_resume_environment.ipynb first")

print("\n⚠️  REMINDER: This notebook is read-only.")
print("   Failure labs in Module 4 will modify secrets and cause disruptions.")
print("   Only run failure labs in TEST/NON-PRODUCTION environments.")

ok("Environment check complete")


## 1. Configuration

Load and validate configuration from environment variables.


In [ ]:
import os
from shared._validation import ok, warn
from shared._cloud_helpers import get_cloud_provider, get_region

# Required configuration
required_vars = ["NAMESPACE", "CLUSTER_NAME"]

print("### Loading Configuration\n")

config = {}
missing = []

for var in required_vars:
    value = os.environ.get(var, "").strip()
    if not value:
        missing.append(var)
    config[var] = value

if missing:
    raise RuntimeError(f"❌ Missing required environment variables: {', '.join(missing)}\n"
                      f"💡 Copy env-samples/workshop.env.example to your .env file and fill in values")

# Optional but recommended
config["HELM_RELEASE"] = os.environ.get("HELM_RELEASE", "langsmith")
config["LANGSMITH_DOMAIN"] = os.environ.get("LANGSMITH_DOMAIN", "")

namespace = config["NAMESPACE"]

# Show cloud provider info
provider = get_cloud_provider()
region = get_region()

print(f"Cloud Provider: {provider.upper()}")
print(f"Region: {region}")
print(f"Namespace: {namespace}")
print(f"Helm Release: {config['HELM_RELEASE']}")

if config["LANGSMITH_DOMAIN"]:
    print(f"LangSmith Domain: {config['LANGSMITH_DOMAIN']}")

ok("Configuration loaded")


## 2. Cluster State Snapshot

Capture a complete snapshot of all resources in the namespace. This is your "before" picture.


In [ ]:
from shared._shell import run
import json

print("### Capturing Cluster State Snapshot\n")

# Get all resources
print("1. Collecting all resources...")
result = run(
    ["kubectl", "get", "all", "-n", namespace, "-o", "wide"],
    check=False,
    stream=False
)

if result.returncode == 0:
    snapshot_file = baseline_dir / "all-resources.txt"
    with open(snapshot_file, "w") as f:
        f.write(result.stdout)
    ok(f"Saved resource snapshot to {snapshot_file.name}")
    print(f"   Resources captured: {len(result.stdout.splitlines())} lines")
else:
    warn("Could not capture resource snapshot")

# Get all resources as YAML (more detailed)
print("\n2. Collecting detailed YAML...")
result = run(
    ["kubectl", "get", "all", "-n", namespace, "-o", "yaml"],
    check=False,
    stream=False
)

if result.returncode == 0:
    yaml_file = baseline_dir / "all-resources.yaml"
    with open(yaml_file, "w") as f:
        f.write(result.stdout)
    ok(f"Saved detailed YAML to {yaml_file.name}")
else:
    warn("Could not capture detailed YAML")


## 3. Key Deployments Description

Get detailed information about key deployments.


In [ ]:
print("### Describing Key Deployments\n")

# Get list of deployments
result = run(
    ["kubectl", "get", "deployments", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    deployments = json.loads(result.stdout)
    deployment_items = deployments.get("items", [])
    
    if deployment_items:
        ok(f"Found {len(deployment_items)} deployment(s)")
        
        # Describe each deployment
        for deployment in deployment_items:
            name = deployment.get("metadata", {}).get("name", "")
            print(f"\n3. Describing deployment: {name}")
            
            result = run(
                ["kubectl", "describe", "deployment", name, "-n", namespace],
                check=False,
                stream=False
            )
            
            if result.returncode == 0:
                desc_file = baseline_dir / f"deployment-{name}.txt"
                with open(desc_file, "w") as f:
                    f.write(result.stdout)
                print(f"   ✅ Saved description to {desc_file.name}")
            else:
                warn(f"Could not describe deployment {name}")
    else:
        warn("No deployments found")
else:
    warn("Could not list deployments")


## 4. Recent Events

Capture recent events sorted by timestamp. Events often contain the first clues about what's happening.


In [ ]:
print("### Collecting Recent Events\n")

# Get events sorted by timestamp
result = run(
    ["kubectl", "get", "events", "-n", namespace, "--sort-by='.lastTimestamp'"],
    check=False,
    stream=False
)

if result.returncode == 0:
    events_file = baseline_dir / "events.txt"
    with open(events_file, "w") as f:
        f.write(result.stdout)
    ok(f"Saved events to {events_file.name}")
    
    # Count events by type
    lines = result.stdout.strip().split("\n")
    if len(lines) > 1:  # Header + events
        event_count = len(lines) - 1
        print(f"   Captured {event_count} event(s)")
        
        # Show last few events
        if event_count > 0:
            print("\n   Last 5 events:")
            for line in lines[-5:]:
                if line.strip():
                    print(f"   {line}")
    else:
        print("   No events found (this is normal for a healthy cluster)")
else:
    warn("Could not collect events")


## 5. Resource Usage

Capture resource usage (CPU, memory) if metrics are available.


In [ ]:
print("### Collecting Resource Usage\n")

# Top pods
print("1. Checking pod resource usage...")
result = run(
    ["kubectl", "top", "pods", "-n", namespace],
    check=False,
    stream=False
)

if result.returncode == 0:
    top_pods_file = baseline_dir / "top-pods.txt"
    with open(top_pods_file, "w") as f:
        f.write(result.stdout)
    ok(f"Saved pod resource usage to {top_pods_file.name}")
    print(result.stdout)
else:
    warn("Could not get pod resource usage (metrics server may not be available)")
    print("   💡 This is OK - metrics are optional for baseline collection")

# Top nodes (if available)
print("\n2. Checking node resource usage...")
result = run(
    ["kubectl", "top", "nodes"],
    check=False,
    stream=False
)

if result.returncode == 0:
    top_nodes_file = baseline_dir / "top-nodes.txt"
    with open(top_nodes_file, "w") as f:
        f.write(result.stdout)
    ok(f"Saved node resource usage to {top_nodes_file.name}")
    print(result.stdout)
else:
    warn("Could not get node resource usage (metrics server may not be available)")


## 6. Canonical Diagnostics Script

**This is the most important step.** Run the official LangChain diagnostics script that Support expects.

The script captures:
- Pod logs (all containers)
- Events (sorted by timestamp)
- Resource usage (CPU, memory)
- Configuration (deployments, services, ingress)
- Storage (PVCs, storage classes)
- Network (services, endpoints)


In [ ]:
import urllib.request
import subprocess

print("### Running Canonical Diagnostics Script\n")

# URL to the canonical script
script_url = "https://raw.githubusercontent.com/langchain-ai/helm/main/charts/langsmith/scripts/get_k8s_debugging_info.sh"
script_path = baseline_dir / "get_k8s_debugging_info.sh"

print(f"1. Downloading script from: {script_url}")
try:
    urllib.request.urlretrieve(script_url, script_path)
    ok(f"Downloaded script to {script_path.name}")
    
    # Make executable
    script_path.chmod(0o755)
    
    # Run the script
    print(f"\n2. Running diagnostics script for namespace: {namespace}")
    print("   (This may take a few minutes...)")
    
    result = run(
        [str(script_path), namespace],
        check=False,
        stream=True  # Stream output so user can see progress
    )
    
    if result.returncode == 0:
        ok("Diagnostics script completed successfully")
        
        # The script creates a tarball - find it
        diagnostics_tarball = None
        for file in baseline_dir.parent.iterdir():
            if file.name.startswith("langsmith-debug-") and file.suffix == ".tar.gz":
                diagnostics_tarball = file
                break
        
        if diagnostics_tarball:
            # Move it to our baseline directory
            target_path = baseline_dir / diagnostics_tarball.name
            diagnostics_tarball.rename(target_path)
            ok(f"Diagnostics bundle saved to: {target_path.name}")
            print(f"   Size: {target_path.stat().st_size / 1024 / 1024:.2f} MB")
        else:
            warn("Could not find diagnostics tarball (check script output above)")
    else:
        warn(f"Diagnostics script returned non-zero exit code: {result.returncode}")
        print("   Check the output above for errors")
        print("   💡 The script may still have collected useful information")
        
except urllib.request.URLError as e:
    warn(f"Could not download diagnostics script: {e}")
    print("   💡 You can download it manually and run it:")
    print(f"      curl -O {script_url}")
    print(f"      chmod +x get_k8s_debugging_info.sh")
    print(f"      ./get_k8s_debugging_info.sh {namespace}")
except Exception as e:
    warn(f"Error running diagnostics script: {e}")


## 7. Basic Health Check

Perform a basic HTTP check to verify the LangSmith endpoint is reachable.


In [ ]:
import requests
import urllib3

# Disable SSL warnings for self-signed certs
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("### Testing Endpoint Reachability\n")

# Determine endpoint URL
if config["LANGSMITH_DOMAIN"]:
    test_url = f"https://{config['LANGSMITH_DOMAIN']}"
else:
    # Try to get from ingress
    result = run(
        ["kubectl", "get", "ingress", "-n", namespace, "-o", "json"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0:
        ingresses = json.loads(result.stdout)
        for ingress in ingresses.get("items", []):
            rules = ingress.get("spec", {}).get("rules", [])
            for rule in rules:
                host = rule.get("host", "")
                if host:
                    test_url = f"https://{host}"
                    break
    else:
        test_url = None

if test_url:
    print(f"Testing: {test_url}")
    try:
        response = requests.get(test_url, allow_redirects=True, verify=False, timeout=10)
        
        health_file = baseline_dir / "endpoint-health.txt"
        with open(health_file, "w") as f:
            f.write(f"URL: {test_url}\n")
            f.write(f"Status Code: {response.status_code}\n")
            f.write(f"Response Headers:\n{json.dumps(dict(response.headers), indent=2)}\n")
        
        if response.status_code in [200, 302, 401, 403]:
            ok(f"Endpoint is reachable (HTTP {response.status_code})")
            print(f"   Response saved to {health_file.name}")
        else:
            warn(f"Endpoint returned unexpected status: {response.status_code}")
    except requests.exceptions.SSLError:
        warn("SSL verification failed (may be self-signed certificate)")
        print("   💡 This is OK for testing. In production, use proper TLS certificates.")
    except requests.exceptions.RequestException as e:
        warn(f"Could not reach endpoint: {e}")
        print("   💡 Endpoint may still be provisioning or DNS not configured")
else:
    warn("No endpoint URL available for testing")
    print("   💡 Set LANGSMITH_DOMAIN in your .env file to test endpoint reachability")


## 8. What Good Looks Like

Quick validation checks to confirm the baseline is healthy.


In [ ]:
from shared._validation import ok, warn

print("### Quick Health Validation\n")

# Check pod status
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

healthy_pods = 0
unhealthy_pods = []

if result.returncode == 0:
    pods = json.loads(result.stdout)
    for pod in pods.get("items", []):
        name = pod.get("metadata", {}).get("name", "")
        phase = pod.get("status", {}).get("phase", "")
        container_statuses = pod.get("status", {}).get("containerStatuses", [])
        
        is_ready = True
        for cs in container_statuses:
            if not cs.get("ready", False):
                is_ready = False
                break
        
        if phase == "Running" and is_ready:
            healthy_pods += 1
        else:
            unhealthy_pods.append((name, phase, is_ready))
    
    if unhealthy_pods:
        warn(f"Found {len(unhealthy_pods)} pod(s) that are not healthy:")
        for name, phase, ready in unhealthy_pods:
            print(f"   - {name}: phase={phase}, ready={ready}")
    else:
        ok(f"All {healthy_pods} pod(s) are healthy and ready")
else:
    warn("Could not check pod status")

# Check for CrashLoopBackOff
if unhealthy_pods:
    crash_loops = [name for name, phase, _ in unhealthy_pods if phase == "CrashLoopBackOff"]
    if crash_loops:
        warn(f"Found {len(crash_loops)} pod(s) in CrashLoopBackOff:")
        for name in crash_loops:
            print(f"   - {name}")
        print("   💡 Check pod logs to understand why they're crashing")

# Check for Pending pods
pending = [name for name, phase, _ in unhealthy_pods if phase == "Pending"]
if pending:
    warn(f"Found {len(pending)} pod(s) in Pending state:")
    for name in pending:
        print(f"   - {name}")
    print("   💡 Check events and resource availability")

print("\n### Baseline Summary\n")
print(f"✅ Baseline captured at: {timestamp}")
print(f"📁 Baseline directory: {baseline_dir}")
print(f"📊 Resources captured:")
print(f"   - Cluster state snapshot")
print(f"   - Deployment descriptions")
print(f"   - Recent events")
print(f"   - Resource usage (if available)")
print(f"   - Canonical diagnostics bundle")
print(f"   - Endpoint health check")

ok("Baseline collection complete!")
print("\n💡 Use this baseline as your reference point for all failure labs.")
print("   Compare future diagnostics to this baseline to identify what changed.")
